In [6]:
from embedder import Embedder
from gitsource import GithubRepositoryDataReader

## Q1. Embedding query

In [ ]:
query = "How does approximate nearest neighbor search work?"

embed = Embedder()
query_vector = embed.encode(query)
query_vector[0]

np.float64(-0.020582036807885073)

### Question: The embedder returns a vector of 384 numbers. What's the first value (v[0])?
### Answer: -0.02

## Q2. Cosine similarity

In [8]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [9]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [13]:
sqlitesearch_document = next(document for document in documents if document["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md")
sqlitesearch_document

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [ ]:
sqlitesearch_vector = embed.encode(sqlitesearch_document["content"])
query_vector.dot(sqlitesearch_vector)

np.float64(0.3610702803026059)

### Question: Take the page `02-vector-search/lessons/07-sqlitesearch-vector.ml`, embed its `content`, and compute the cosine similarity with the query vector from Q1. What do you get?
### Answer: 0.37

In [ ]:
from tqdm.auto import tqdm
batch_size = 50
vectors = []
for i in tqdm(range(0, len(documents), batch_size)):
    batch = documents[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    vectors.extend(batch_vectors)
import numpy as np
X = np.array(vectors)
scores = X.dot(query)